In [7]:
import sys
sys.path.insert(0, '../')

In [8]:
from get_data import Data
import backtest
from combinations import sim_conditions
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [11]:
data=Data()

In [12]:
'''
下載資料
1. price:股價相關
2. report:財報相關
'''
data.get("price:close")

company_symbol,1101,1102,1103,1104,1108,1109,1110,1201,1203,1210,...,9944,9945,9946,9949,9950,9951,9955,9958,9960,9962
date,,,,,,,,,,,,,,,,,,,,,
2000-01-04,7.5067,5.3558,5.5019,2.6105,5.4752,5.4740,6.2886,6.5692,6.4118,1.7259,...,NaN,1.5950,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2000-01-05,7.7020,5.7189,5.5304,2.6507,5.6920,5.5840,6.2573,7.0125,6.4685,1.7978,...,NaN,1.6852,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2000-01-06,7.3766,5.7734,5.7584,2.7042,5.6920,5.7159,6.4450,7.4961,6.6387,1.8377,...,NaN,1.7605,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2000-01-07,7.3549,5.7734,5.8155,2.6908,5.6920,5.7379,6.4450,8.0200,6.6955,1.8457,...,NaN,1.7605,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2000-01-10,7.5935,5.8097,5.9580,2.6908,5.6107,5.6719,6.3512,8.5439,6.8941,1.8537,...,NaN,1.7605,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2023-12-25,34.3500,40.3000,18.8000,29.3500,15.7000,18.1500,18.7500,19.0000,53.2000,57.7000,...,20.15,36.2500,20.15,17.60,22.8,75.3,24.20,168.0,27.30,18.45
2023-12-26,34.5000,40.5500,18.8000,29.4000,15.7500,18.1000,18.5500,19.0500,53.5000,57.4000,...,20.20,36.4000,20.15,18.05,22.6,76.8,24.20,167.0,27.25,18.30
2023-12-27,34.6500,40.9000,18.9000,29.5500,15.8000,18.1000,18.7500,19.0500,54.7000,57.3000,...,20.15,36.8000,20.40,17.90,22.4,75.8,24.45,167.0,27.10,18.20


# 策略回測

In [13]:
# data = Data()
roe = data.get("report:roe")
roe_slices = data.get("report:roe").divide_slice(4, ascending=False)
roe_1, roe_2, roe_3, roe_4 = [
    roe_slices['Quantile_{}'.format(i)] for i in range(1, 5)
]

conditions = {
    'roe_1': roe_1,
    'roe_2': roe_2,
    'roe_3': roe_3,
    'roe_4': roe_4
}

In [14]:
# 執行多條件回測
report_collection = sim_conditions(
    conditions=conditions,
    resample='M',
    data=data
)

# 繪製各組合的累積報酬曲線
fig = report_collection.plot_creturns()
fig.show()

# 取得各組合的統計指標
stats = report_collection.get_stats()
print(stats)

# 繪製統計指標長條圖
fig = report_collection.plot_stats(mode='bar')
fig.show()


Backtesting progress: 100%|██████████| 4/4 [00:24<00:00,  6.15s/condition]


                  roe_1      roe_2      roe_3      roe_4
CAGR           0.120382   0.102490   0.068465   0.075234
daily_sharpe  20.650000  22.080000  29.740000  32.510000
max_drawdown  -0.512147  -0.581260  -0.583351  -0.624983
avg_drawdown  -0.072140  -0.102792  -0.117834  -0.174152
win_ratio      0.428505   0.476336   0.413366   0.397502
ytd            0.382136   0.273705   0.247936   0.214854


In [15]:
# 從 data 取出 pb 因子
pb = data.get("report:pb")

# 獲取股價、因子資料
factor_df_dict = {
    "roe": roe,
    "pb": pb
}
factor_asc_dict = {
    "roe": False,   # 排序方式：False → 降冪（高 ROE 優先）
    "pb": True      # 排序方式：True  → 升冪（低 PB 優先）
}

In [16]:
import os
from factor_analysis import factor_analysis_two_factor_AA

# 執行兩因子切割並回傳結果
result_AA = factor_analysis_two_factor_AA(
    factor_name_list = ["roe", "pb"],
    factor_asc_dict   = factor_asc_dict,
    all_factor_df_dict= factor_df_dict
)

# 輸出每個分位數的結果到 CSV
directory = "Quantile_AA"
if not os.path.exists(directory):
    os.makedirs(directory)

for key, df in result_AA.items():
    df.to_csv(f"{directory}\\{key}_on_example_AA.csv")


In [17]:
report_collection_AA = sim_conditions(
    conditions = result_AA, 
    resample = 'M',       
    data = data        
)

# 繪製累積報酬率曲線
fig = report_collection_AA.plot_creturns()
fig.show()

# 列印各組統計指標
stats = report_collection_AA.get_stats()
print(stats)

# 繪製統計指標長條圖
fig = report_collection_AA.plot_stats(mode='bar')
fig.show()

Backtesting progress: 100%|██████████| 4/4 [00:13<00:00,  3.43s/condition]


              Quantile_1  Quantile_2  Quantile_3  Quantile_4
CAGR            0.172684    0.143339    0.054370   -0.042908
daily_sharpe   15.370000   17.440000   37.580000   36.700000
max_drawdown   -0.314948   -0.560188   -0.665382   -0.896274
avg_drawdown   -0.030879   -0.100737   -0.142483   -0.730613
win_ratio       0.445813    0.546619    0.362246    0.346572
ytd             0.389280    0.314802    0.186026    0.194674


In [18]:
from factor_analysis import factor_analysis_two_factor_AND

result_two_factor = factor_analysis_two_factor_AND(
    factor_name_list    = ["roe", "pb"],
    factor_asc_dict     = factor_asc_dict,
    all_factor_df_dict  = factor_df_dict,
)
report_collection_tf = sim_conditions(
    conditions = result_two_factor,
    resample = 'M',
    data = data
)

# 繪製累積報酬率曲線
fig = report_collection_tf.plot_creturns()
fig.show()

# 取得各組合統計指標並印出
stats = report_collection_tf.get_stats()
print(stats)

# 繪製統計指標長條圖
fig = report_collection_tf.plot_stats(mode='bar')
fig.show()


Backtesting progress: 100%|██████████| 4/4 [00:16<00:00,  4.22s/condition]


              Quantile_1  Quantile_2  Quantile_3  Quantile_4
CAGR            0.128828    0.158481    0.031912   -0.052303
daily_sharpe   17.930000   17.200000   57.990000   30.430000
max_drawdown   -0.292950   -0.559689   -0.699383   -0.915012
avg_drawdown   -0.038763   -0.081771   -0.277666   -0.792792
win_ratio       0.328643    0.540015    0.445111    0.372835
ytd             0.401237    0.291672    0.252797    0.210829


In [19]:
# 1. 計算 AA 方法──每個分位的股票數
aa_counts = {q: len(df) for q, df in result_AA.items()}
total_aa  = sum(aa_counts.values())

print("AA 方法各分位股票數：")
for q, cnt in aa_counts.items():
    print(f"  {q}：{cnt} 支")
print(f"AA 方法總共選出 {total_aa} 支股票\n")


# 2. 計算 AND 方法──每個組合的股票數
and_counts = {g: len(df) for g, df in result_two_factor.items()}
total_and  = sum(and_counts.values())

print("AND 方法各組合股票數：")
for g, cnt in and_counts.items():
    print(f"  {g}：{cnt} 支")
print(f"AND 方法總共選出 {total_and} 支股票")


AA 方法各分位股票數：
  Quantile_1：95 支
  Quantile_2：95 支
  Quantile_3：95 支
  Quantile_4：95 支
AA 方法總共選出 380 支股票

AND 方法各組合股票數：
  Quantile_1：95 支
  Quantile_2：95 支
  Quantile_3：95 支
  Quantile_4：95 支
AND 方法總共選出 380 支股票


In [20]:
all_factor_df_dict = {
    'pb': data.get("report:pb"),
    'ev_ebitda': data.get("report:ev_ebitda")
}

factor_ratio_dict = {
    'pb': 0.7,
    'ev_ebitda': 0.3
}

factor_asc_dict = {
    'pb': False,
    'ev_ebitda': False
}
quantile = 10

method = 'ranked'

In [21]:
from factor_analysis import cal_factor_sum_df_interpolated
all_quantile = cal_factor_sum_df_interpolated(
                ['pb','ev_ebitda'],
                factor_ratio_dict,
                factor_asc_dict,
                quantile,
                method,
                all_factor_df_dict)

quantile_1 = all_quantile['Quantile_1']

In [22]:
quantile_1

company_symbol,1101,1102,1103,1104,1108,1109,1110,1201,1203,1210,...,9944,9945,9946,9949,9950,9951,9955,9958,9960,9962
2000-05-15,True,False,False,False,False,False,False,True,False,False,...,False,False,False,False,False,False,False,False,False,False
2000-08-31,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
2000-11-15,True,True,False,False,False,False,False,True,False,False,...,False,False,False,False,False,False,False,False,False,False
2001-03-31,False,False,False,False,False,True,False,True,False,False,...,False,False,False,False,False,False,False,False,False,False
2001-05-15,False,False,False,False,False,False,False,False,False,False,...,True,False,False,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2022-11-15,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
2023-03-31,False,False,False,False,False,False,False,False,False,False,...,False,False,False,True,False,False,False,False,False,False
2023-08-31,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,True,False,False
2023-11-15,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False


In [23]:
quantile_1

company_symbol,1101,1102,1103,1104,1108,1109,1110,1201,1203,1210,...,9944,9945,9946,9949,9950,9951,9955,9958,9960,9962
2000-05-15,True,False,False,False,False,False,False,True,False,False,...,False,False,False,False,False,False,False,False,False,False
2000-08-31,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
2000-11-15,True,True,False,False,False,False,False,True,False,False,...,False,False,False,False,False,False,False,False,False,False
2001-03-31,False,False,False,False,False,True,False,True,False,False,...,False,False,False,False,False,False,False,False,False,False
2001-05-15,False,False,False,False,False,False,False,False,False,False,...,True,False,False,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2022-11-15,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
2023-03-31,False,False,False,False,False,False,False,False,False,False,...,False,False,False,True,False,False,False,False,False,False
2023-08-31,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,True,False,False
2023-11-15,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
